In [1]:
!pip install transformers torchvision scikit-learn pandas

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from transformers import AutoTokenizer, AutoModel
import numpy as np

# 1. 12 Knee Problems
ABNORMALITIES = [
    "ACL_Tear", "PCL_Tear", "MCL_Injury", "LCL_Injury",
    "Medial_Meniscus_Tear", "Lateral_Meniscus_Tear",
    "Tendon_Injury", "Medial_Cartilage_Defect", "Lateral_Cartilage_Defect",
    "Patella_Defect", "Joint_Fluid_Collection", "Bone_Contusion"
]
NUM_CLASSES = len(ABNORMALITIES)

print(f"Total target abnormalities to detect: {NUM_CLASSES}")

# 2. 500 Patient Samples (MRI Images + Doctor Reports) Setup
class KneeDataset(Dataset):
    def __init__(self, total_patients=500, tokenizer=None):
        self.total_patients = total_patients
        self.tokenizer = tokenizer

        # Clinical doctor sample reports
        self.reports_templates = [
            "Severe ACL fiber tear noted with joint fluid accumulation. Cruciate ligaments abnormal.",
            "Horizontal tear in medial meniscus. Cartilage thinning and joint effusion observed.",
            "Normal knee scan. Cruciate and collateral ligaments intact. No bone injury.",
            "Patellar cartilage wear, bone contusion in lateral femoral condyle. MCL sprain."
        ]
        # 500 patients ki 12 diseases  labels (0 = ledu, 1 = undi)
        np.random.seed(42)
        self.ground_truth = np.random.choice([0, 1], size=(total_patients, 12), p=[0.75, 0.25]).astype(np.float32)

    def __len__(self):
        return self.total_patients

    def __getitem__(self, idx):
        # 1 MRI Image (3 channels, 224x224 standard medical view)
        image = torch.randn(3, 224, 224)

        # 1 Doctor Report text
        report = self.reports_templates[idx % len(self.reports_templates)]
        text_tokens = self.tokenizer(
            report, padding="max_length", truncation=True, max_length=64, return_tensors="pt"
        )

        labels = torch.tensor(self.ground_truth[idx])
        return {
            "image": image,
            "input_ids": text_tokens["input_ids"].squeeze(0),
            "attention_mask": text_tokens["attention_mask"].squeeze(0),
            "labels": labels
        }

# 3. Model Architecture (Eyes: ResNet for Image | Brain: ClinicalBERT for Text)
class MultimodalKneeAI(nn.Module):
    def __init__(self):
        super(MultimodalKneeAI, self).__init__()

        # Vision Part
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.vision_extractor = nn.Sequential(*list(resnet.children())[:-1])
        self.vision_proj = nn.Linear(512, 128)

        # Text Part (Bio_ClinicalBERT)
        self.tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
        self.text_extractor = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
        for param in self.text_extractor.parameters():
            param.requires_grad = False  # Faster execution
        self.text_proj = nn.Linear(768, 128)

        # Fusion Part (Image + Text kalipede classifier)
        self.classifier = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, NUM_CLASSES)
        )

    def forward(self, img, input_ids, attention_mask):
        # Vision features
        v = self.vision_extractor(img).squeeze(-1).squeeze(-1)
        v_feat = self.vision_proj(v)  # [Batch, 128]

        # Text features
        t = self.text_extractor(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        t_feat = self.text_proj(t)    # [Batch, 128]

        # Fusion (Image 128 + Text 128 = 256)
        combined = torch.cat((v_feat, t_feat), dim=1)
        predictions = self.classifier(combined)
        return predictions

# 4. Training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using hardware:", device)

model = MultimodalKneeAI().to(device)
dataset = KneeDataset(total_patients=500, tokenizer=model.tokenizer)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

print("\n--- Model Training Started (500 Samples) ---")
model.train()
for epoch in range(2):
    epoch_loss = 0.0
    for step, batch in enumerate(dataloader):
        imgs = batch["image"].to(device)
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(imgs, input_ids, attn_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}/2 - Loss: {epoch_loss/len(dataloader):.4f}")

# Model save
torch.save(model.state_dict(), "knee_multimodal_model.pth")
print("\nSuccess! Model saved as: knee_multimodal_model.pth")

Total target abnormalities to detect: 12
Using hardware: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 128MB/s]


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

...

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Model Training Started (500 Samples) ---


model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 1/2 - Loss: 0.5851
Epoch 2/2 - Loss: 0.5678

Success! Model saved as: knee_multimodal_model.pth


In [3]:
import torch
import torch.nn.functional as F

def test_single_patient(mri_image_tensor, doctor_report_text):
    """
    Live Working Demo: MRI scan + Text note theeskoni
    12 diseases lo edi undo predict chesthundi.
    """
    model.eval() # Evaluation mode

    # 1. Text tokenize cheyadam
    text_tokens = model.tokenizer(
        doctor_report_text,
        padding="max_length",
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

    # Device setup
    img = mri_image_tensor.unsqueeze(0).to(device)
    input_ids = text_tokens["input_ids"].to(device)
    attn_mask = text_tokens["attention_mask"].to(device)

    # 2. Model Prediction
    with torch.no_grad():
        raw_outputs = model(img, input_ids, attn_mask)
        # Probabilities ga convert cheyadam (0% to 100%)
        probabilities = torch.sigmoid(raw_outputs).squeeze(0).cpu().numpy()

    print("\n" + "="*50)
    print("PATIENT MULTIMODAL DIAGNOSIS REPORT")
    print("="*50)
    print(f"Clinical Note: \"{doctor_report_text}\"")
    print("-"*50)
    print(f"{'Abnormality':<28} | {'Probability':<10} | {'Status'}")
    print("-"*50)

    detected_count = 0
    for disease_name, prob in zip(ABNORMALITIES, probabilities):
        prob_pct = prob * 100
        # 50% kante ekkuva unte Detected ga consider chestham
        status = "POSITIVE (Detected)" if prob_pct >= 50 else "Negative"
        print(f"{disease_name:<28} | {prob_pct:>6.2f}%    | {status}")
        if prob_pct >= 50:
            detected_count += 1

    print("="*50)
    print(f"Total Detected Abnormalities: {detected_count}/12")

# --- LIVE TEST RUN ---
# Sample MRI slice & Clinical Report
test_mri_image = torch.randn(3, 224, 224)
test_clinical_note = "Patient sustained twisting trauma during sports. Severe pain and joint effusion noted. Suspected complete ACL tear with medial meniscus instability."

test_single_patient(test_mri_image, test_clinical_note)


PATIENT MULTIMODAL DIAGNOSIS REPORT
Clinical Note: "Patient sustained twisting trauma during sports. Severe pain and joint effusion noted. Suspected complete ACL tear with medial meniscus instability."
--------------------------------------------------
Abnormality                  | Probability | Status
--------------------------------------------------
ACL_Tear                     |  30.44%    | Negative
PCL_Tear                     |  25.31%    | Negative
MCL_Injury                   |  18.04%    | Negative
LCL_Injury                   |  21.12%    | Negative
Medial_Meniscus_Tear         |  19.56%    | Negative
Lateral_Meniscus_Tear        |  28.30%    | Negative
Tendon_Injury                |  33.40%    | Negative
Medial_Cartilage_Defect      |  17.70%    | Negative
Lateral_Cartilage_Defect     |  22.81%    | Negative
Patella_Defect               |  22.70%    | Negative
Joint_Fluid_Collection       |  20.36%    | Negative
Bone_Contusion               |  18.78%    | Negative
Total D

In [5]:
!ls


knee_multimodal_model.pth  sample_data
